# Fake Review Detection - Model Training

In [10]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
import os

In [11]:
data_dir = 'Data'

df_train = pd.read_csv(os.path.join(data_dir, 'train.csv'), sep=';')
df_test = pd.read_csv(os.path.join(data_dir, 'test.csv'), sep=';')
df_val = pd.read_csv(os.path.join(data_dir, 'validation.csv'), sep=';')

print(f'Train: {len(df_train)}, Test: {len(df_test)}, Validation: {len(df_val)}')

## Prepare Features

In [12]:
X_train = df_train['text']
y_train = df_train['ai_generated']

X_test = df_test['text']
y_test = df_test['ai_generated']

## TF-IDF Vectorization

In [13]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

## Hyperparameter Tuning + Logistic Regression

In [14]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear'],
    'class_weight': ['balanced']
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)
grid.fit(X_train_tfidf, y_train)

model = grid.best_estimator_
print(f'Best params: {grid.best_params_}')
print(f'Best CV F1:  {grid.best_score_:.4f}')
print(f'Vocabulary size: {len(tfidf.get_feature_names_out())}')

Fitting 5 folds for each of 8 candidates, totalling 40 fits


c:\Users\behna\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Best params: {'C': 10, 'class_weight': 'balanced', 'penalty': 'l2', 'solver': 'liblinear'}
Best CV F1:  0.8487
Vocabulary size: 10000


## Evaluation on Train Set

In [15]:
y_pred_train = model.predict(X_train_tfidf)

print('=== Train Set ===')
print(f'Accuracy:  {accuracy_score(y_train, y_pred_train):.4f}')
print(f'F1-Score:  {f1_score(y_train, y_pred_train):.4f}')
print(f'Precision: {precision_score(y_train, y_pred_train):.4f}')
print(f'Recall:    {recall_score(y_train, y_pred_train):.4f}')

=== Train Set ===
Accuracy:  0.9927
F1-Score:  0.9581
Precision: 0.9196
Recall:    1.0000


## Evaluation on Test Set

In [16]:
y_pred = model.predict(X_test_tfidf)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)

print('=== Test Set ===')
print(f'Accuracy:  {acc:.4f}')
print(f'F1-Score:  {f1:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print()
print('Classification Report:')
print(classification_report(y_test, y_pred))

from sklearn.metrics import roc_auc_score
y_prob = model.predict_proba(X_test_tfidf)[:, 1]
print(f'ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}')

=== Test Set ===
Accuracy:  0.9751
F1-Score:  0.8614
Precision: 0.8099
Recall:    0.9200

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      8182
           1       0.81      0.92      0.86       750

    accuracy                           0.98      8932
   macro avg       0.90      0.95      0.92      8932
weighted avg       0.98      0.98      0.98      8932



## Top 10 Markers for AI (Fake) and Human (Real)

In [17]:
feature_names = np.array(tfidf.get_feature_names_out())
coefs = model.coef_[0]

top_ai_idx = np.argsort(coefs)[-10:][::-1]
top_human_idx = np.argsort(coefs)[:10]

print("=== Top 10 Markers for AI-Generated (Fake) ===")
for i in top_ai_idx:
    print(f"  {feature_names[i]:25s} weight: {coefs[i]:.4f}")

print("\n=== Top 10 Markers for Human-Written (Real) ===")
for i in top_human_idx:
    print(f"  {feature_names[i]:25s} weight: {coefs[i]:.4f}")

=== Top 10 Markers for AI-Generated (Fake) ===
  bit                       weight: 10.0643
  completely                weight: 9.2596
  every                     weight: 9.1617
  but the                   weight: 8.9200
  during                    weight: 8.8808
  incredibly                weight: 8.1784
  digital                   weight: 8.1318
  incredible                weight: 8.1286
  feels                     weight: 8.0717
  two days                  weight: 7.7524

=== Top 10 Markers for Human-Written (Real) ===
  br                        weight: -13.1376
  good                      weight: -10.9834
  not                       weight: -10.1169
  nice                      weight: -9.8734
  so                        weight: -9.5444
  pretty                    weight: -8.3818
  product                   weight: -7.7204
  got                       weight: -7.3815
  would                     weight: -7.0240
  purchased                 weight: -6.7508


## Leave-One-Category-Out Evaluation

In [ ]:
results_loc = []
categories = sorted(df_train['category'].unique())

for held_out in categories:
    train_sub = df_train[df_train['category'] != held_out]
    test_sub = df_test[df_test['category'] == held_out]

    tfidf_loc = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
    X_tr = tfidf_loc.fit_transform(train_sub['text'])
    X_te = tfidf_loc.transform(test_sub['text'])

    m = LogisticRegression(C=10, class_weight='balanced', solver='liblinear', max_iter=1000, random_state=42)
    m.fit(X_tr, train_sub['ai_generated'])
    pred = m.predict(X_te)

    results_loc.append({
        'category': held_out,
        'accuracy': accuracy_score(test_sub['ai_generated'], pred),
        'precision': precision_score(test_sub['ai_generated'], pred, zero_division=0),
        'recall': recall_score(test_sub['ai_generated'], pred, zero_division=0),
        'f1': f1_score(test_sub['ai_generated'], pred, zero_division=0),
        'samples': len(test_sub)
    })

loc_df = pd.DataFrame(results_loc)
avg_row = pd.DataFrame([{
    'category': 'AVERAGE',
    'accuracy': loc_df['accuracy'].mean(),
    'precision': loc_df['precision'].mean(),
    'recall': loc_df['recall'].mean(),
    'f1': loc_df['f1'].mean(),
    'samples': loc_df['samples'].sum()
}])
pd.concat([loc_df, avg_row], ignore_index=True)